# grok-008 · QLoRA 规模与显存（学习向）

> **对应深入层 B（效率工程）**：0.5B 全精度 LoRA 已在 001–007 练过；本课补上 **「穷人卡上的真·规模」**。
>
> **你在学什么**
> 1. **4bit NF4 量化** 为何能把 3B 塞进 T4  
> 2. **QLoRA** = 冻住 4bit 基座 + 只训 LoRA  
> 3. 读 **peak VRAM / 步耗时**，而不是只看 loss  
> 4. 双 T4 可见时：先确认硬件，再理解「量化模型 + 多卡」并不总是简单 DP
>
> **本课模型**：`Qwen/Qwen2.5-3B-Instruct`（比 7B 更稳；7B 作为可选挑战，OOM 则自动降级说明）  
> **不学什么**：刷 GSM8K 高分（留给 009/010）

## 学习检查（跑完自问）
- 4bit 基座参数是否 `requires_grad=False`？可训的只有谁？  
- peak mem 大概多少 GB？若换 7B 失败原因是什么？  


In [ ]:
# 【步骤】双卡可见：不做 CUDA_VISIBLE_DEVICES 锁定（本课要看见 2×T4）
import os
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("CUDA_VISIBLE_DEVICES", os.environ.get("CUDA_VISIBLE_DEVICES"))


In [ ]:
# 【步骤】环境与硬件体检
import os, gc, time, json, math, platform, traceback
from pathlib import Path

import torch

OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)

print("python", platform.python_version())
print("torch", torch.__version__)
assert torch.cuda.is_available(), "需要 GPU"
n = torch.cuda.device_count()
print("device_count", n)
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {torch.cuda.get_device_name(i)}  {p.total_memory/1e9:.1f}GB  cap={torch.cuda.get_device_capability(i)}")

# T4 = SM 7.5：优先 fp16 计算 dtype（无原生 bf16 tensor core）
DEVICE = torch.device("cuda:0")  # 主设备：默认第一张可见 GPU
print("primary", DEVICE)


In [ ]:
# 【步骤】探测 bitsandbytes / peft / transformers（QLoRA 三件套）
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

print("transformers", transformers.__version__)
try:
    import bitsandbytes as bnb
    print("bitsandbytes", getattr(bnb, "__version__", "?"))
    HAS_BNB = True
except Exception as e:
    print("bitsandbytes 不可用:", repr(e)[:200])
    HAS_BNB = False

# peft 偶发 torchao 探测炸掉：直接关掉该分发路径
try:
    import peft.tuners.lora.torchao as peft_torchao
    peft_torchao.is_torchao_available = lambda: False
    print("patched peft torchao -> False")
except Exception as e:
    print("torchao patch skip", e)


In [ ]:
# 【步骤】选择模型：优先 3B QLoRA；失败则记录原因（学习「规模边界」）
CANDIDATES = [
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",  # 回退：更省显存
]

def try_load_4bit(model_id: str):
    """4bit NF4：QLoRA 论文同款思路——权重压缩存放，计算时再展开。"""
    assert HAS_BNB, "无 bitsandbytes 无法走标准 QLoRA"
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",           # NormalFloat4
        bnb_4bit_use_double_quant=True,     # 二次量化再省一点
        bnb_4bit_compute_dtype=torch.float16,
    )
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # device_map 由 accelerate 自动切层；双卡时可能拆到两张卡（推理友好，训练要小心）
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_cfg,
        device_map="auto",
        trust_remote_code=True,
    )
    return tok, model

loaded_id = None
tokenizer = model = None
errors = []
for mid in CANDIDATES:
    try:
        print("try load", mid)
        torch.cuda.empty_cache(); gc.collect()
        tokenizer, model = try_load_4bit(mid)
        loaded_id = mid
        print("OK", mid)
        break
    except Exception as e:
        errors.append({"model": mid, "error": repr(e)[:400]})
        print("FAIL", mid, repr(e)[:200])

assert loaded_id is not None, errors
print("using", loaded_id)


In [ ]:
# 【步骤】注入 LoRA：只训练低秩旁路；4bit 基座保持冻结
model = prepare_model_for_kbit_training(model)  # kbit 训练稳定性（embedding 输入要梯度等）
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()  # 用算力换显存：激活重计算
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

cand = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
present = sorted({n.split(".")[-1] for n,_ in model.named_modules() if n.split(".")[-1] in cand})
print("lora targets", present)

lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=present,
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

# 统计：确认「可训参数 << 总参数」
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
print(f"trainable {n_train:,} / {n_all:,} = {100*n_train/max(1,n_all):.4f}%")


In [ ]:
# 【步骤】构造极小 SFT batch：本课目的是测显存/速度，不是刷分
SAMPLES = [
    ("What is 17 * 19? Reply with the number only.", "323"),
    ("Write a Python function add(a,b) returning the sum. Code only.", "def add(a, b):\n    return a + b\n"),
    ("用一句话解释 LoRA。", "LoRA 冻结原模型，只训练注入线性层的低秩适配矩阵。"),
] * 8

def to_batch(pairs, max_len=512):
    texts = []
    for q, a in pairs:
        messages = [{"role":"user","content":q},{"role":"assistant","content":a}]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
    batch = tokenizer(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    return batch

BATCH = to_batch(SAMPLES[:4])
print({k: v.shape for k,v in BATCH.items()})


In [ ]:
# 【步骤】短训几步：记录 peak VRAM 与 step 时间（效率工程的核心数据）
from torch.optim import AdamW

# 量化+device_map 模型：输入放到模型首参数所在设备
def model_input_device(m):
    try:
        return next(m.parameters()).device
    except StopIteration:
        return DEVICE

dev = model_input_device(model)
print("param device example", dev)

opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
STEPS = 12
model.train()
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
losses = []
for step in range(STEPS):
    batch = {k: v.to(dev) for k,v in BATCH.items()}
    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100  # -100：CE 忽略 pad
    opt.zero_grad(set_to_none=True)
    with torch.cuda.amp.autocast(dtype=torch.float16):
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=labels)
        loss = out.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    opt.step()
    losses.append(float(loss.detach().float().cpu()))
    if step % 3 == 0 or step == STEPS-1:
        print(f"step {step:02d}/{STEPS} loss={losses[-1]:.4f}")

elapsed = time.perf_counter() - t0
peak_gb = torch.cuda.max_memory_allocated() / 1e9
sec_per_step = elapsed / STEPS
print(f"elapsed_s={elapsed:.1f} sec/step={sec_per_step:.2f} peak_mem_gb={peak_gb:.2f}")


In [ ]:
# 【步骤】可选：尝试 7B 仅「加载」测边界（失败不视为本课失败）
seven_b_report = {"tried": False}
try:
    mid7 = "Qwen/Qwen2.5-7B-Instruct"
    print("challenge load", mid7)
    torch.cuda.empty_cache(); gc.collect()
    # 注意：不训练，只看能否 4bit 加载
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    m7 = AutoModelForCausalLM.from_pretrained(
        mid7, quantization_config=bnb_cfg, device_map="auto", trust_remote_code=True,
    )
    seven_b_report = {
        "tried": True, "ok": True, "model": mid7,
        "devices": sorted({str(p.device) for p in m7.parameters()}),
    }
    del m7
    torch.cuda.empty_cache()
    print("7B 4bit load OK", seven_b_report)
except Exception as e:
    seven_b_report = {"tried": True, "ok": False, "model": "Qwen/Qwen2.5-7B-Instruct", "error": repr(e)[:500]}
    print("7B load failed (expected on tight VRAM):", seven_b_report["error"][:200])


In [ ]:
# 【步骤】写出报告：把「规模/显存/速度」变成可复盘数字
report = {
    "notebook": "grok-008-qlora-scale-bench",
    "phase": "deep_B_efficiency",
    "device_count": torch.cuda.device_count(),
    "device_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "loaded_model": loaded_id,
    "load_errors": errors,
    "trainable_params": n_train,
    "total_params_reported": n_all,
    "trainable_pct": 100.0 * n_train / max(1, n_all),
    "steps": STEPS,
    "loss_start": losses[0],
    "loss_end": losses[-1],
    "sec_per_step": sec_per_step,
    "peak_mem_gb": peak_gb,
    "seven_b_challenge": seven_b_report,
    "takeaways_zh": [
        "QLoRA=4bit冻基座+LoRA可训旁路",
        "看 peak_mem 与 sec/step，不只看 loss",
        "7B 是否可加载取决于序列/batch/并行方式",
    ],
}
path = OUT / "grok008_results.json"
path.write_text(json.dumps(report, indent=2, ensure_ascii=False))
print(json.dumps(report, indent=2, ensure_ascii=False)[:2000])
print("DONE grok-008")


## 学习检查清单
- 你应能回答：NF4 / double quant 各解决什么问题？  
- 为何 QLoRA 可训参数比例通常 <2%？  
- 双 T4 的 32GB **不等于** 单卡 32GB 连续显存——体现在哪里？  
- 下一步：009 用 **GSM8K** 建「不可 hack」的硬尺子。  
